# Deliverables Quality Inspection & Regulatory Validation Suite
### Visual Bounding Box Inspector & Regulatory Compliance Auditor (10/10 Checks)
**Course**: CSC15006 - Introduction to Natural Language Processing (NLP)  
**Official Standard**: `guide/Quy_định_đồ_án.pdf` (FIT HCMUS)  

---

This notebook allows instant verification and bounding box visualization of all book folders in `output/` without re-running the heavy PDF extraction pipeline.

**Features**:
1. **Automated Regulatory Compliance Auditor**: Audits all 24 book folders against the 10 official criteria from `guide/Quy_định_đồ_án.pdf`.
2. **Interactive & Batch Bounding Box Visualizer**: Renders high-resolution bounding box overlays (Green for Chữ Nôm, Blue for Quốc ngữ, Red for margin guide) directly in notebook outputs.
3. **Single Page Inspector**: Allows inspecting any specific page of any book with a single function call.


In [ ]:
# Environment Setup and Core Module Imports
import os
import sys
import json
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import cv2
import numpy as np
import pandas as pd
import openpyxl
import fitz  # PyMuPDF
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.config import PDF_DIR, OUTPUT_DIR
from core.preprocess import cv2_imread_unicode

print(f"Workspace Root : {PROJECT_ROOT}")
print(f"PDF Directory  : {PDF_DIR}")
print(f"Output Dir     : {OUTPUT_DIR}")


In [ ]:
# Automated Regulatory Compliance Auditor (10/10 Criteria)
def validate_ppocr_format(label_path: Path) -> bool:
    if not label_path.exists():
        return False
    lines = [l.strip() for l in label_path.read_text(encoding="utf-8").splitlines() if l.strip()]
    if not lines:
        return True
    for line in lines:
        parts = line.split("\t")
        if len(parts) != 2:
            return False
        try:
            boxes = json.loads(parts[1])
            for b in boxes:
                if len(b.get("points", [])) != 4:
                    return False
        except Exception:
            return False
    return True

book_dirs = sorted([p for p in OUTPUT_DIR.iterdir() if p.is_dir() and (p / "ch").exists() and not p.name.startswith(".")])

audit_results = []
print("=" * 115)
print(f"AUDITING COMPLIANCE (10/10 CRITERIA) FOR ALL {len(book_dirs)} DELIVERABLE FOLDERS")
print("=" * 115)

for b_dir in book_dirs:
    stem = b_dir.name
    ch_dir = b_dir / "ch"
    vi_dir = b_dir / "vi"
    misc_dir = b_dir / "misc"
    pdf_file = b_dir / f"{stem}.pdf"
    csv_file = b_dir / f"{stem}.csv"
    xlsx_file = b_dir / f"{stem}.xlsx"
    
    src_pdf = PDF_DIR / f"{stem}.pdf"
    src_pages = 0
    if src_pdf.exists():
        try:
            doc = fitz.open(src_pdf)
            src_pages = len(doc)
            doc.close()
        except Exception:
            src_pages = 0
            
    ch_imgs = set(p.name for p in ch_dir.glob("*.jpg"))
    vi_imgs = set(p.name for p in vi_dir.glob("*.jpg"))
    misc_imgs = set(p.name for p in misc_dir.glob("*.jpg"))
    total_preserved = len(ch_imgs.union(vi_imgs).union(misc_imgs))
    
    checks = {}
    checks["1_Page_Conservation"] = (total_preserved == src_pages) if src_pages > 0 else True
    checks["2_Subdir_ch"] = (ch_dir / "Label.txt").exists() and (ch_dir / "fileState.txt").exists()
    checks["3_Subdir_vi"] = (vi_dir / "Label.txt").exists() and (vi_dir / "fileState.txt").exists()
    checks["4_Subdir_misc"] = misc_dir.is_dir()
    checks["5_Replicated_PDF"] = pdf_file.exists() and pdf_file.stat().st_size > 0
    checks["6_ch_PPOCRLabel_4pt"] = validate_ppocr_format(ch_dir / "Label.txt")
    checks["7_vi_PPOCRLabel_4pt"] = validate_ppocr_format(vi_dir / "Label.txt")
    
    try:
        df_c = pd.read_csv(csv_file, encoding="utf-8-sig")
        checks["8_CSV_BOM_4Cols"] = list(df_c.columns[:4]) == ["img_id", "sentence_id", "SinoNom_Char", "ChuQN_txt"]
        num_pairs = len(df_c)
    except Exception:
        checks["8_CSV_BOM_4Cols"] = False
        num_pairs = 0
        
    try:
        wb = openpyxl.load_workbook(xlsx_file, data_only=True)
        ws = wb.active
        checks["9_XLSX_ARGB_Colors"] = any(
            c.font and c.font.color and c.font.color.rgb
            for row in ws.iter_rows(min_row=2, max_row=min(ws.max_row, 15))
            for c in row
        ) or (ws.max_row <= 1)
    except Exception:
        checks["9_XLSX_ARGB_Colors"] = False
        
    checks["10_fileState_1"] = True
    for s_file in [ch_dir / "fileState.txt", vi_dir / "fileState.txt"]:
        if s_file.exists():
            for line in s_file.read_text(encoding="utf-8").splitlines():
                if line.strip() and not line.endswith("\t1"):
                    checks["10_fileState_1"] = False
                    break
                    
    pass_count = sum(1 for v in checks.values() if v)
    all_ok = pass_count == 10
    
    audit_results.append({
        "Book_Title": stem[:35],
        "PDF_Pages": src_pages,
        "Conserved": total_preserved,
        "ch_Imgs": len(ch_imgs),
        "vi_Imgs": len(vi_imgs),
        "misc_Imgs": len(misc_imgs),
        "Pairs": num_pairs,
        "Score": f"{pass_count}/10",
        "Compliance": "PASS [10/10]" if all_ok else f"FAIL [{pass_count}/10]"
    })

scorecard_df = pd.DataFrame(audit_results)
print(scorecard_df.to_string(index=False))
print("=" * 115)
passed_num = sum(1 for r in audit_results if r["Compliance"] == "PASS [10/10]")
print(f"\nTOTAL BOOKS AUDITED: {len(audit_results)} | 100% COMPLIANT: {passed_num} / {len(audit_results)}")
if passed_num == len(audit_results):
    print("STATUS: ALL DELIVERABLES FULLY COMPLY WITH QUY_DINH_DO_AN.PDF (10/10)")


In [ ]:
# Interactive & Batch Bounding Box Visualizer
def render_page_with_bboxes(book_title: str, img_name: str, output_dir: Path = OUTPUT_DIR, show_plot: bool = True, fig_size: Tuple[int, int] = (10, 14)) -> Optional[np.ndarray]:
    """
    Renders tight bounding boxes onto manuscript scan:
    - Green polygons: Chữ Hán Nôm (from ch/Label.txt)
    - Blue polygons : Chữ Quốc ngữ (from vi/Label.txt)
    - Red guideline : Margin delimiter (x = 260)
    """
    book_dir = output_dir / book_title
    if not book_dir.exists():
        print(f"Error: Book directory not found: {book_dir}")
        return None
        
    img_p = book_dir / "ch" / img_name
    if not img_p.exists():
        img_p = book_dir / "vi" / img_name
    if not img_p.exists():
        print(f"Error: Image not found: {img_name}")
        return None
        
    img = cv2_imread_unicode(img_p)
    if img is None:
        return None
        
    h, w, _ = img.shape
    overlay = img.copy()
    
    # Load Hán Nôm boxes
    ch_boxes = []
    ch_lbl = book_dir / "ch" / "Label.txt"
    if ch_lbl.exists():
        for l in ch_lbl.read_text(encoding="utf-8").splitlines():
            if l.strip():
                parts = l.split("\t")
                if Path(parts[0]).name == img_name:
                    try: ch_boxes = json.loads(parts[1])
                    except Exception: pass
                    break
                    
    # Load Quốc ngữ boxes
    vi_boxes = []
    vi_lbl = book_dir / "vi" / "Label.txt"
    if vi_lbl.exists():
        for l in vi_lbl.read_text(encoding="utf-8").splitlines():
            if l.strip():
                parts = l.split("\t")
                if Path(parts[0]).name == img_name:
                    try: vi_boxes = json.loads(parts[1])
                    except Exception: pass
                    break
                    
    # Draw Red Margin Guide at x=260 if image width > 400
    if w > 400:
        cv2.line(overlay, (260, 0), (260, h), (0, 0, 255), 2)
        cv2.putText(overlay, "Red Margin Guide x=260", (265, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        
    # Draw Chữ Hán Nôm boxes in Green (0, 200, 0)
    for idx, b in enumerate(ch_boxes, 1):
        pts = np.array(b["points"], np.int32).reshape((-1, 1, 2))
        cv2.polylines(overlay, [pts], isClosed=True, color=(0, 200, 0), thickness=3)
        p1 = b["points"][0]
        cv2.putText(overlay, f"N{idx}", (max(0, p1[0] - 40), min(h - 5, p1[1] + 18)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 180, 0), 2)
        
    # Draw Chữ Quốc ngữ boxes in Blue (255, 120, 0)
    for idx, b in enumerate(vi_boxes, 1):
        pts = np.array(b["points"], np.int32).reshape((-1, 1, 2))
        cv2.polylines(overlay, [pts], isClosed=True, color=(255, 120, 0), thickness=3)
        p1 = b["points"][0]
        cv2.putText(overlay, f"Q{idx}", (max(0, p1[0] - 40), min(h - 5, p1[1] + 18)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (240, 90, 0), 2)
        
    # Summary Legend Banner at Top
    cv2.rectangle(overlay, (15, 15), (580, 85), (255, 255, 255), -1)
    cv2.rectangle(overlay, (15, 15), (580, 85), (0, 0, 0), 2)
    cv2.putText(overlay, f"Green: Chu Nom ({len(ch_boxes)} boxes)", (25, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 160, 0), 2)
    cv2.putText(overlay, f"Blue: Quoc Ngu ({len(vi_boxes)} boxes)", (25, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (220, 80, 0), 2)
    
    if show_plot:
        plt.figure(figsize=fig_size)
        plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        plt.title(f"{book_title}\nImage: {img_name} | Nom: {len(ch_boxes)} | QN: {len(vi_boxes)}", fontsize=11, fontweight="bold")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
        
    return overlay

def visualize_book_samples(book_title: str, max_pages: int = 1, output_dir: Path = OUTPUT_DIR):
    book_dir = output_dir / book_title
    if not book_dir.exists():
        return
    ch_lbl = book_dir / "ch" / "Label.txt"
    target_imgs = []
    if ch_lbl.exists():
        for l in ch_lbl.read_text(encoding="utf-8").splitlines():
            if l.strip():
                parts = l.split("\t")
                boxes = json.loads(parts[1]) if len(parts) == 2 else []
                if len(boxes) > 0:
                    target_imgs.append(Path(parts[0]).name)
                if len(target_imgs) >= max_pages:
                    break
    for img_name in target_imgs:
        render_page_with_bboxes(book_title, img_name, output_dir=output_dir, show_plot=True)

# Render Sample Bounding Boxes for Books
sample_books_to_show = [
    "Ca Trù Thê Cách, AB 160, Nôm_Y",
    "Ca Trù Thế Cách, AB 431, Nôm_Y",
    "Ca Trù Thế Cách, AB 456, Nôm_Y",
    "Ca Trù Thể Cách, Nôm_Y",
    "ThuDạLữHoàiNgâm_秋夜旅懷吟",
    "TìBàQuốcÂmTânTruyện_琵琶國音新傳_1"
]

print("RENDERING SAMPLE BOUNDING BOX OVERLAYS FOR BOOKS:")
for b_name in sample_books_to_show:
    if (OUTPUT_DIR / b_name).exists():
        visualize_book_samples(b_name, max_pages=1)


In [ ]:
# Single Page Custom Inspector
# Example: Inspect any page of any book directly by title and image filename
sample_book = "Ca Trù Thê Cách, AB 160, Nôm_Y"
sample_image = "Ca Trù Thê Cách, AB 160, Nôm_Y.006.jpg"

print(f"Inspecting specific manuscript page: {sample_book} -> {sample_image}")
render_page_with_bboxes(sample_book, sample_image, show_plot=True)
